# Chess.com PubAPI — Connection Test

A step-by-step notebook that verifies connectivity to the Chess.com public API.
Each step prints the raw JSON response so you can inspect the data structure
before building analysis on top of it.

**Steps covered:**
1. Fetch player profile
2. Fetch list of monthly game archives
3. Fetch all games from the most recent archive month
4. Extract and print moves from one sample game's PGN

In [1]:
import json
import re
import requests

# ── Configuration ─────────────────────────────────────────────────────────────
USERNAME: str = "OrangeMutante"
BASE_URL: str = "https://api.chess.com/pub"
HEADERS: dict[str, str] = {"User-Agent": "my-chess-analysis/1.0 contact@email.com"}

## Step 1 — Fetch Player Profile

Endpoint: `GET /pub/player/{username}`  
Returns basic account info: country, join date, avatar, ratings, etc.

In [2]:
def get_player_profile(username: str) -> dict:
    """Fetches the public profile for a Chess.com player."""
    url: str = f"{BASE_URL}/player/{username}"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    return response.json()


profile: dict = get_player_profile(USERNAME)
print(json.dumps(profile, indent=2))

{
  "avatar": "https://images.chesscomfiles.com/uploads/v1/user/149547101.795dd138.200x200o.42e6f0a12e63.jpg",
  "player_id": 149547101,
  "@id": "https://api.chess.com/pub/player/orangemutante",
  "url": "https://www.chess.com/member/OrangeMutante",
  "name": "Kenji Tetard",
  "username": "orangemutante",
  "followers": 4,
  "country": "https://api.chess.com/pub/country/CH",
  "last_online": 1779870677,
  "joined": 1627659749,
  "status": "premium",
  "is_streamer": false,
  "verified": false,
  "league": "Champion",
  "streaming_platforms": []
}


## Step 2 — Fetch Monthly Game Archives

Endpoint: `GET /pub/player/{username}/games/archives`  
Returns a list of URLs, one per month that contains at least one game.

In [3]:
def get_game_archives(username: str) -> list[str]:
    """Returns the list of monthly archive URLs for a player."""
    url: str = f"{BASE_URL}/player/{username}/games/archives"
    response = requests.get(url, headers=HEADERS)
    response.raise_for_status()
    data: dict = response.json()
    return data["archives"]


archives: list[str] = get_game_archives(USERNAME)
print(json.dumps(archives, indent=2))
print(f"\nTotal archive months found: {len(archives)}")

[
  "https://api.chess.com/pub/player/orangemutante/games/2025/09",
  "https://api.chess.com/pub/player/orangemutante/games/2025/10",
  "https://api.chess.com/pub/player/orangemutante/games/2025/11",
  "https://api.chess.com/pub/player/orangemutante/games/2025/12",
  "https://api.chess.com/pub/player/orangemutante/games/2026/01",
  "https://api.chess.com/pub/player/orangemutante/games/2026/02",
  "https://api.chess.com/pub/player/orangemutante/games/2026/03",
  "https://api.chess.com/pub/player/orangemutante/games/2026/04",
  "https://api.chess.com/pub/player/orangemutante/games/2026/05"
]

Total archive months found: 9


## Step 3 — Fetch Games from Most Recent Month

Endpoint: `GET /pub/player/{username}/games/{YYYY}/{MM}`  
We use the last URL in the archives list (most recent month).
Each game object contains metadata plus the full PGN string.

In [4]:
def get_games_from_archive(archive_url: str) -> list[dict]:
    """Fetches all games from a single monthly archive URL."""
    response = requests.get(archive_url, headers=HEADERS)
    response.raise_for_status()
    data: dict = response.json()
    return data["games"]


latest_archive_url: str = archives[-1]
print(f"Fetching from: {latest_archive_url}\n")

games: list[dict] = get_games_from_archive(latest_archive_url)
print(f"Games in most recent month: {len(games)}\n")

# Print the first game to reveal the full object structure
print("Sample game object (first entry):")
print(json.dumps(games[0], indent=2))

Fetching from: https://api.chess.com/pub/player/orangemutante/games/2026/05

Games in most recent month: 250

Sample game object (first entry):
{
  "url": "https://www.chess.com/game/live/168085449150",
  "pgn": "[Event \"Live Chess\"]\n[Site \"Chess.com\"]\n[Date \"2026.05.01\"]\n[Round \"-\"]\n[White \"OrangeMutante\"]\n[Black \"magickids12\"]\n[Result \"1-0\"]\n[CurrentPosition \"6k1/2Q5/1P6/7p/7p/P6P/5PP1/6K1 b - - 0 37\"]\n[Timezone \"UTC\"]\n[ECO \"C44\"]\n[ECOUrl \"https://www.chess.com/openings/Ponziani-Opening-Jaenisch-Counterattack-4.d4-exd4-5.e5\"]\n[UTCDate \"2026.05.01\"]\n[UTCTime \"13:12:04\"]\n[WhiteElo \"630\"]\n[BlackElo \"617\"]\n[TimeControl \"60\"]\n[Termination \"OrangeMutante won by resignation\"]\n[StartTime \"13:12:04\"]\n[EndDate \"2026.05.01\"]\n[EndTime \"13:13:51\"]\n[Link \"https://www.chess.com/game/live/168085449150\"]\n\n1. e4 {[%clk 0:00:59.7]} 1... e5 {[%clk 0:00:59]} 2. Nf3 {[%clk 0:00:59.3]} 2... Nc6 {[%clk 0:00:58.3]} 3. c3 {[%clk 0:00:59.2]} 3... 

## Step 4 — Extract Moves from a Sample Game's PGN

PGN (Portable Game Notation) is a plain-text chess format.  
Header lines start with `[` and contain metadata (players, date, result).  
The move text follows, e.g.: `1. e4 e5 2. Nf3 Nc6 ...`  
Chess.com may include inline clock annotations like `{[%clk 0:05:00]}` — these are stripped.

In [5]:
def extract_moves_from_pgn(pgn: str) -> list[str]:
    """
    Parses a PGN string and returns the ordered list of moves.

    Strips header lines (starting with '['), inline comments ({...}),
    move-number tokens (e.g. '1.' or '1...'), and result tokens.
    """
    lines: list[str] = pgn.strip().splitlines()

    # Keep only lines that are part of the move text (not headers)
    move_lines: list[str] = [
        line for line in lines
        if not line.startswith("[") and line.strip()
    ]
    move_text: str = " ".join(move_lines)

    # Remove inline comments such as clock annotations: {[%clk 0:10:00]}
    move_text = re.sub(r"\{[^}]*\}", "", move_text)

    result_tokens: set[str] = {"1-0", "0-1", "1/2-1/2", "*"}
    moves: list[str] = []

    for token in move_text.split():
        if token in result_tokens:
            continue
        # Move-number tokens end with one or more dots (e.g. '1.' or '1...')
        if token.rstrip(".").isdigit():
            continue
        moves.append(token)

    return moves


sample_pgn: str = games[0].get("pgn", "")
moves: list[str] = extract_moves_from_pgn(sample_pgn)

print("Raw PGN snippet (first 500 chars):")
print(sample_pgn[:500])
print("\nExtracted move list:")
print(moves)

Raw PGN snippet (first 500 chars):
[Event "Live Chess"]
[Site "Chess.com"]
[Date "2026.05.01"]
[Round "-"]
[White "OrangeMutante"]
[Black "magickids12"]
[Result "1-0"]
[CurrentPosition "6k1/2Q5/1P6/7p/7p/P6P/5PP1/6K1 b - - 0 37"]
[Timezone "UTC"]
[ECO "C44"]
[ECOUrl "https://www.chess.com/openings/Ponziani-Opening-Jaenisch-Counterattack-4.d4-exd4-5.e5"]
[UTCDate "2026.05.01"]
[UTCTime "13:12:04"]
[WhiteElo "630"]
[BlackElo "617"]
[TimeControl "60"]
[Termination "OrangeMutante won by resignation"]
[StartTime "13:12:04"]
[EndDate "

Extracted move list:
['e4', 'e5', 'Nf3', 'Nc6', 'c3', 'Nf6', 'd4', 'exd4', 'e5', 'Qe7', 'cxd4', 'Ng4', 'h3', 'Nh6', 'Bxh6', 'gxh6', 'Be2', 'Bg7', 'O-O', 'O-O', 'Nc3', 'Qb4', 'Qd2', 'Re8', 'a3', 'Qb5', 'Bxb5', 'a6', 'Bxc6', 'bxc6', 'b4', 'd5', 'exd6', 'cxd6', 'Rfe1', 'Be6', 'd5', 'cxd5', 'Nxd5', 'Bxd5', 'Rxe8+', 'Rxe8', 'Qxd5', 'Rd8', 'Re1', 'Be5', 'Qc6', 'Rf8', 'Nxe5', 'dxe5', 'Rxe5', 'f6', 'Re7', 'Rf7', 'Qc7', 'Rxe7', 'Qxe7', 'Kh8', 'Qxf6+', 'Kg8', 'Qe7', 'K

## Summary

Aggregated results from the four steps above.

In [6]:
# ── Final summary ─────────────────────────────────────────────────────────────
print(f"Player            : {profile.get('username', USERNAME)}")
print(f"Archive months    : {len(archives)}")
print(f"Games (latest mo.): {len(games)}")
print(f"Moves in sample   : {len(moves)}")
print(f"\nMove list: {' '.join(moves)}")

Player            : orangemutante
Archive months    : 9
Games (latest mo.): 250
Moves in sample   : 73

Move list: e4 e5 Nf3 Nc6 c3 Nf6 d4 exd4 e5 Qe7 cxd4 Ng4 h3 Nh6 Bxh6 gxh6 Be2 Bg7 O-O O-O Nc3 Qb4 Qd2 Re8 a3 Qb5 Bxb5 a6 Bxc6 bxc6 b4 d5 exd6 cxd6 Rfe1 Be6 d5 cxd5 Nxd5 Bxd5 Rxe8+ Rxe8 Qxd5 Rd8 Re1 Be5 Qc6 Rf8 Nxe5 dxe5 Rxe5 f6 Re7 Rf7 Qc7 Rxe7 Qxe7 Kh8 Qxf6+ Kg8 Qe7 Kh8 Qa7 Kg8 Qxa6 h5 Qa7 h4 Qc7 h6 b5 h5 b6
